# Tutorial: LineageResolver Scanpy Walkthrough

This notebook demonstrates the v1 one-call workflow on the bundled fixture data.
It covers both ambient modes: `raw10x` and fallback.


## What You Will Run

1. Load fixture AnnData and config/evidence files.
2. Run `lineageresolver.infer(...)` using `raw_10x_path`.
3. Inspect posterior, abstention, and evidence columns.
4. Run fallback mode (no `raw_10x_path`) and compare diagnostics.


In [ ]:
from pathlib import Path

import anndata as ad
import lineageresolver as lr

DATA_DIR = Path('tests/data')
adata = ad.read_h5ad(DATA_DIR / 'filtered_small.h5ad')
config_path = DATA_DIR / 'configs/cytotoxic_adjudication_v1.yaml'
raw10x_path = DATA_DIR / 'raw10x_small/raw_feature_bc_matrix'
evidence_path = DATA_DIR / 'evidence_small.tsv'

adata.shape


In [ ]:
adata_raw = lr.infer(
    adata=adata,
    task_config=config_path,
    raw_10x_path=raw10x_path,
    evidence_table=evidence_path,
    tau=0.9,
    return_posteriors=True,
    inplace=False,
)

adata_raw.obs[[
    'lineageresolver_label_map',
    'lineageresolver_label_call',
    'lineageresolver_max_p',
    'lineageresolver_uncertainty_entropy',
]].head()


In [ ]:
posterior_cols = ['lineageresolver_p_NK', 'lineageresolver_p_gdT', 'lineageresolver_p_abT']

print('Ambient mode:', adata_raw.uns['lineageresolver_ambient_estimation_mode'])
print('Posterior row-sum check:', adata_raw.obs[posterior_cols].sum(axis=1).round(6).head().tolist())
print('Label-call counts:')
print(adata_raw.obs['lineageresolver_label_call'].value_counts())


In [ ]:
adata_fallback = lr.infer(
    adata=adata,
    task_config=config_path,
    evidence_table=evidence_path,
    tau=0.9,
    return_posteriors=True,
    inplace=False,
)

{
    'raw_mode': adata_raw.uns['lineageresolver_ambient_estimation_mode'],
    'fallback_mode': adata_fallback.uns['lineageresolver_ambient_estimation_mode'],
}


## Interpreting Results

- `lineageresolver_label_map` is the top posterior class.
- `lineageresolver_label_call` applies abstention (`tau`).
- `lineageresolver_max_p` and entropy help rank confidence.
- `lineageresolver_ambient_estimation_mode` confirms whether ambient came from `raw10x` or fallback.
